In [ ]:

# ============================================================
# C1NC0 - VALIDAÇÃO, QUALIDADE E EXPLORAÇÃO DO FAKE.BR
# ============================================================
#
# Objetivo:
#
# Utilizar diretamente o mesmo dataset Fake.br-Corpus usado no
# notebook Dataset_Novo.ipynb e executar uma análise em etapas:
#
# 1. baixar o Fake.br-Corpus;
# 2. localizar os arquivos de metadados;
# 3. montar o mesmo DataFrame utilizado em Dataset_Novo.ipynb;
# 4. obter as URLs da coluna "link";
# 5. validar formato, acesso, redirecionamentos e códigos HTTP;
# 6. separar registros com e sem erro de acesso;
# 7. preparar os dados para análise;
# 8. identificar incoerências por regras de qualidade;
# 9. identificar anomalias estatísticas com IQR;
# 10. identificar anomalias multivariadas com Isolation Forest;
# 11. encontrar padrões com estatísticas descritivas e K-Means;
# 12. usar PCA para visualizar os clusters em duas dimensões;
# 13. fazer perguntas orientadoras sobre os dados;
# 14. gerar gráficos para exploração e data storytelling;
# 15. comparar bibliotecas de visualização;
# 16. registrar explicitamente que correlação não implica
#     causalidade;
# 17. gerar arquivos finais para continuidade do projeto.
#
# IMPORTANTE:
#
# O Fake.br-Corpus possui os rótulos "fake" e "true".
#
# Esses rótulos podem ser utilizados para perguntas descritivas
# sobre o corpus, mas o projeto C1NC0 NÃO deve transformar esses
# resultados em uma afirmação de que uma feature isolada prova
# que uma notícia é falsa, verdadeira ou confiável.
#
# Uma anomalia também NÃO significa notícia falsa.
#
# Uma incoerência indica que determinado registro merece revisão
# de qualidade ou de interpretação.
#
# Uma correlação mostra associação estatística, não causa.
#
# URL acessível NÃO significa conteúdo confiável.
#
# URL inacessível NÃO significa conteúdo falso.
# ============================================================


# Biblioteca principal para trabalhar com tabelas/DataFrames.
import pandas as pd

# Biblioteca para operações numéricas.
import numpy as np

# Biblioteca utilizada para fazer requisições HTTP.
import requests

# Biblioteca base para criação e exportação de gráficos.
import matplotlib.pyplot as plt

# Biblioteca de visualização estatística construída sobre Matplotlib.
import seaborn as sns

# Ferramenta da biblioteca padrão para interpretar URLs.
from urllib.parse import urlparse

# Permite validar várias URLs simultaneamente.
from concurrent.futures import ThreadPoolExecutor, as_completed

# Biblioteca utilizada para medir o tempo das requisições.
import time

# Biblioteca para trabalhar com arquivos e diretórios.
import os

# Biblioteca para executar o comando git clone a partir do Python.
import subprocess

# Biblioteca para compactar a pasta de gráficos ao final.
import shutil

# Padroniza as features antes do K-Means.
from sklearn.preprocessing import StandardScaler

# Algoritmo de clustering usado para encontrar padrões sem rótulos.
from sklearn.cluster import KMeans

# Técnica de redução de dimensionalidade usada para visualização.
from sklearn.decomposition import PCA

# Métrica que ajuda a comparar diferentes quantidades de clusters.
from sklearn.metrics import silhouette_score

# Algoritmo não supervisionado para detecção de anomalias.
from sklearn.ensemble import IsolationForest

# Ferramenta do Google Colab utilizada para baixar
# os arquivos produzidos ao final do notebook.
from google.colab import files


# Define um tema simples para os gráficos estatísticos.
sns.set_theme(
    style="whitegrid"
)


# Mostra todas as colunas quando necessário.
pd.set_option(
    "display.max_columns",
    None
)


# Mostra uma mensagem para confirmar que a célula foi executada.
print(
    "Ambiente preparado com sucesso!"
)


In [ ]:

# ============================================================
# CONFIGURAÇÃO DO DATASET
# ============================================================


# Endereço oficial do repositório Fake.br-Corpus utilizado
# também no notebook Dataset_Novo.ipynb.
URL_REPOSITORIO = (
    "https://github.com/roneysco/Fake.br-Corpus.git"
)


# Pasta onde o Google Colab armazenará o dataset.
BASE_DATASET = (
    "/content/Fake.br-Corpus"
)


# Mostra as configurações para facilitar a conferência.
print(
    "Repositório:",
    URL_REPOSITORIO
)

print(
    "Pasta local:",
    BASE_DATASET
)


In [ ]:

# ============================================================
# BAIXAR O FAKE.BR-CORPUS
# ============================================================


# Verifica se o diretório do dataset já existe.
if not os.path.isdir(BASE_DATASET):

    # Executa o equivalente a:
    #
    # git clone https://github.com/roneysco/Fake.br-Corpus.git
    #
    # O parâmetro check=True faz o Python interromper esta célula
    # se o comando git retornar algum erro.
    subprocess.run(
        [
            "git",
            "clone",
            URL_REPOSITORIO,
            BASE_DATASET
        ],
        check=True
    )

    # Informa que o download terminou.
    print(
        "\nDownload do Fake.br-Corpus concluído!"
    )


# Caso a pasta já exista, não tentamos clonar novamente.
else:

    # Mostra que o notebook reutilizará o dataset existente.
    print(
        "Fake.br-Corpus já está disponível em:",
        BASE_DATASET
    )


In [ ]:

# ============================================================
# CONFERIR A ESTRUTURA DO DATASET
# ============================================================


# Percorre os itens existentes na pasta principal do dataset.
for item in os.listdir(BASE_DATASET):

    # Mostra cada pasta ou arquivo encontrado.
    print(item)


In [ ]:

# ============================================================
# LOCALIZAR OS ARQUIVOS DE METADADOS
# ============================================================


# Cria uma lista vazia para armazenar os caminhos dos
# arquivos de metadados encontrados.
arquivos_meta = []


# Percorre recursivamente todas as pastas do Fake.br-Corpus.
for raiz, pastas, arquivos in os.walk(BASE_DATASET):

    # Percorre cada arquivo encontrado na pasta atual.
    for arquivo in arquivos:

        # Monta o caminho completo do arquivo.
        caminho = os.path.join(
            raiz,
            arquivo
        )

        # Mantém somente arquivos encontrados em diretórios
        # cujo nome contém "meta-information".
        if "meta-information" in raiz.lower():

            # Adiciona o caminho à lista.
            arquivos_meta.append(
                caminho
            )


# Ordena a lista para tornar a montagem do DataFrame
# reproduzível entre execuções.
arquivos_meta = sorted(
    arquivos_meta
)


# Mostra quantos arquivos de metadados foram encontrados.
print(
    f"Total de arquivos de metadados: "
    f"{len(arquivos_meta):,}"
)


# Mostra os primeiros 10 caminhos apenas para conferência.
print(
    "\nPrimeiros arquivos encontrados:"
)

for caminho in arquivos_meta[:10]:

    print(caminho)


In [ ]:

# ============================================================
# TRANSFORMAR OS METADADOS EM DATAFRAME
# ============================================================
#
# Esta célula reproduz a lógica usada no Dataset_Novo.ipynb.
#
# Cada arquivo "*-meta.txt" possui uma sequência fixa de
# informações. A lista abaixo define o nome de cada posição.
# ============================================================


# Nomes das 25 informações fornecidas pelos metadados
# do Fake.br-Corpus.
colunas = [
    "autor",
    "link",
    "categoria",
    "data_publicacao",
    "num_tokens",
    "num_palavras",
    "num_types",
    "num_links_internos",
    "num_palavras_maiusculas",
    "num_verbos",
    "num_verbos_subjuntivo_imperativo",
    "num_substantivos",
    "num_adjetivos",
    "num_adverbios",
    "num_verbos_modais",
    "num_pronomes_singular_1_2",
    "num_pronomes_plural_1",
    "num_pronomes",
    "pausality",
    "num_caracteres",
    "comprimento_medio_sentenca",
    "comprimento_medio_palavra",
    "percentual_erros_ortograficos",
    "emotividade",
    "diversidade"
]


# Função responsável por ler um único arquivo de metadados.
def ler_metadado(caminho):

    # Abre o arquivo somente para leitura.
    with open(
        caminho,
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as arquivo:

        # Lê todas as linhas do arquivo.
        #
        # strip() remove espaços e quebras de linha
        # no começo e no final de cada valor.
        linhas = [
            linha.strip()
            for linha in arquivo.readlines()
        ]


    # Caso um arquivo tenha menos posições do que o esperado,
    # completa as posições faltantes com None.
    linhas += [None] * (
        len(colunas) - len(linhas)
    )


    # Mantém somente as 25 posições definidas em "colunas".
    linhas = linhas[
        :len(colunas)
    ]


    # Associa cada nome de coluna ao valor correspondente.
    registro = dict(
        zip(
            colunas,
            linhas
        )
    )


    # Devolve o registro montado.
    return registro


# Lista que receberá uma notícia por posição.
registros = []


# Percorre todos os arquivos de metadados localizados.
for caminho in arquivos_meta:

    # Lê os metadados do arquivo atual.
    dados = ler_metadado(
        caminho
    )


    # Verifica se o arquivo pertence ao conjunto "fake".
    if "fake-meta-information" in caminho.lower():

        # Cria o rótulo da notícia.
        dados["classe"] = "fake"


    # Verifica se o arquivo pertence ao conjunto "true".
    elif "true-meta-information" in caminho.lower():

        # Cria o rótulo da notícia.
        dados["classe"] = "true"


    # Mantém uma opção de segurança para caminhos inesperados.
    else:

        # Marca a classe como desconhecida.
        dados["classe"] = "desconhecida"


    # Guarda o caminho de origem para rastreabilidade.
    #
    # IMPORTANTE:
    # "arquivo_origem" não deve ser usado como feature em
    # treinamento de Machine Learning porque o próprio caminho
    # contém "fake-meta-information" ou "true-meta-information".
    dados["arquivo_origem"] = caminho


    # Adiciona o registro à lista.
    registros.append(
        dados
    )


# Transforma a lista de dicionários em DataFrame.
df = pd.DataFrame(
    registros
)


# Mostra a quantidade de notícias.
print(
    f"Total de notícias: "
    f"{len(df):,}"
)


# Mostra a quantidade de colunas.
print(
    f"Total de campos: "
    f"{len(df.columns)}"
)


# Mostra as primeiras linhas do DataFrame.
display(
    df.head()
)


In [ ]:

# ============================================================
# CONFERIR A COLUNA DE URLs DO FAKE.BR
# ============================================================


# Nome da coluna que contém as URLs das notícias.
#
# Esta é a mesma coluna "link" criada no Dataset_Novo.ipynb.
COLUNA_URL = "link"


# Verifica se a coluna esperada realmente existe.
if COLUNA_URL not in df.columns:

    # Interrompe a execução com uma mensagem clara.
    raise KeyError(
        f"A coluna '{COLUNA_URL}' não foi encontrada."
    )


# Mostra a quantidade total de linhas do dataset.
print(
    "Total de registros:",
    len(df)
)


# Mostra quantos valores da coluna "link" são reconhecidos
# inicialmente pelo pandas como nulos.
print(
    "Links nulos:",
    df[COLUNA_URL].isna().sum()
)


# Mostra as primeiras URLs para conferência visual.
print(
    "\nPrimeiras URLs:"
)

display(
    df[
        [
            "classe",
            COLUNA_URL
        ]
    ].head(10)
)


In [ ]:

# ============================================================
# CONFIGURAÇÕES DA VALIDAÇÃO
# ============================================================


# Número máximo de segundos aguardando uma resposta
# de cada site.
TIMEOUT = 10


# Número máximo de URLs verificadas simultaneamente.
#
# Valores muito altos podem:
#
# - sobrecarregar a conexão;
# - aumentar bloqueios;
# - provocar respostas HTTP 429.
#
# Por isso mantemos um valor conservador.
MAX_WORKERS = 8


# Quantidade de URLs usada no teste rápido antes
# da execução completa.
QUANTIDADE_TESTE = 20


# Nome do arquivo que armazenará somente a validação
# técnica das URLs.
ARQUIVO_CHECKPOINT = (
    "C1NC0_validacao_urls_FakeBr.csv"
)


# Nome do arquivo completo contendo o dataset original
# mais as colunas de validação, qualidade, anomalias e padrões.
ARQUIVO_FINAL = (
    "C1NC0_FakeBr_validado_analisado.csv"
)


# Nome do arquivo que conterá somente registros
# classificados como SEM_ERRO_ACESSO.
ARQUIVO_SEM_ERRO = (
    "C1NC0_FakeBr_urls_sem_erro_acesso.csv"
)


# Nome do arquivo que conterá somente registros
# classificados como COM_ERRO_ACESSO.
ARQUIVO_COM_ERRO = (
    "C1NC0_FakeBr_urls_com_erro_acesso.csv"
)


# Nome do arquivo com registros que apresentaram
# anomalias ou incoerências para revisão.
ARQUIVO_REVISAO = (
    "C1NC0_FakeBr_anomalias_incoerencias.csv"
)


# Nome do resumo das regras de qualidade.
ARQUIVO_RESUMO_QUALIDADE = (
    "C1NC0_resumo_anomalias_incoerencias.csv"
)


# Nome do resumo dos clusters encontrados.
ARQUIVO_CLUSTERS = (
    "C1NC0_perfil_clusters.csv"
)


# Nome do roteiro/resumo de data storytelling.
ARQUIVO_STORYTELLING = (
    "C1NC0_data_storytelling_resumo.csv"
)


# Pasta que receberá os gráficos em PNG.
PASTA_GRAFICOS = (
    "/content/c1nc0_graficos"
)


# Nome do arquivo ZIP contendo os gráficos.
ARQUIVO_GRAFICOS_ZIP = (
    "C1NC0_graficos_storytelling.zip"
)


# Identificação enviada aos servidores durante a requisição.
USER_AGENT = (
    "C1NC0-Residencia-IA/0.4 "
    "(validacao educacional de URLs)"
)


# Cabeçalhos HTTP utilizados nas requisições.
HEADERS = {

    # Identifica o programa que está fazendo a requisição.
    "User-Agent": USER_AGENT,

    # Informa preferência por português brasileiro,
    # aceitando também português e inglês.
    "Accept-Language": (
        "pt-BR,pt;q=0.9,en;q=0.8"
    )
}


# Cria a pasta de gráficos caso ela ainda não exista.
os.makedirs(
    PASTA_GRAFICOS,
    exist_ok=True
)


# Mostra as configurações escolhidas.
print(
    "TIMEOUT:",
    TIMEOUT
)

print(
    "MAX_WORKERS:",
    MAX_WORKERS
)

print(
    "QUANTIDADE_TESTE:",
    QUANTIDADE_TESTE
)

print(
    "Pasta de gráficos:",
    PASTA_GRAFICOS
)


In [ ]:

# ============================================================
# FUNÇÃO - VALIDAR O FORMATO DA URL
# ============================================================


# Recebe uma URL e retorna True ou False.
def formato_url_valido(url):

    # Verifica se o valor é nulo/NaN.
    if pd.isna(url):

        return False


    # Converte o valor para texto.
    url = str(url)


    # Remove espaços do começo e do final.
    url = url.strip()


    # Verifica se, depois da limpeza, a URL ficou vazia.
    if not url:

        return False


    try:

        # Divide a URL em partes:
        #
        # - scheme;
        # - domínio;
        # - caminho;
        # - query;
        # - fragmento.
        partes = urlparse(
            url
        )


        # Aceita somente os protocolos HTTP e HTTPS.
        if partes.scheme not in (
            "http",
            "https"
        ):

            return False


        # A URL precisa possuir domínio/hostname.
        if not partes.netloc:

            return False


        # Se passou pelas verificações anteriores,
        # consideramos o formato válido.
        return True


    # Caso a interpretação da URL gere algum erro.
    except Exception:

        return False


In [ ]:

# ============================================================
# TESTES RÁPIDOS DA FUNÇÃO DE FORMATO
# ============================================================


# Deve retornar True.
print(
    formato_url_valido(
        "https://www.google.com"
    )
)


# Deve retornar False.
print(
    formato_url_valido(
        "isso não é uma url"
    )
)


In [ ]:

# ============================================================
# FUNÇÃO - CLASSIFICAR O STATUS HTTP
# ============================================================


# Recebe o código HTTP e a informação de redirecionamento.
def classificar_http(
    status_code,
    houve_redirect=False
):

    # Códigos de 200 a 299 indicam sucesso HTTP.
    if 200 <= status_code <= 299:

        # Se houve mudança de endereço, registramos
        # como redirecionada mesmo com resposta final 2xx.
        if houve_redirect:

            return "REDIRECIONADA"


        # Resposta normal sem mudança de endereço.
        return "VALIDA"


    # Códigos de 300 a 399 representam redirecionamento.
    if 300 <= status_code <= 399:

        return "REDIRECIONADA"


    # Código 401:
    # o recurso exige autenticação.
    if status_code == 401:

        return "REQUER_AUTENTICACAO"


    # Código 403:
    # o servidor recusou o acesso.
    if status_code == 403:

        return "BLOQUEADA"


    # Código 404:
    # o recurso não foi encontrado.
    if status_code == 404:

        return "NAO_ENCONTRADA"


    # Código 410:
    # o recurso foi removido permanentemente.
    if status_code == 410:

        return "REMOVIDA"


    # Código 429:
    # o servidor recebeu requisições demais.
    if status_code == 429:

        return "RATE_LIMIT"


    # Códigos de 500 a 599:
    # erro no servidor consultado.
    if 500 <= status_code <= 599:

        return "ERRO_SERVIDOR"


    # Outros códigos HTTP não tratados especificamente.
    return "OUTRO_HTTP"


In [ ]:

# ============================================================
# FUNÇÃO - VERIFICAR UMA URL
# ============================================================


# Recebe uma única URL e devolve um dicionário
# com o resultado técnico da verificação.
def verificar_url(url):

    # Cria a estrutura inicial do resultado.
    resultado = {

        # URL que será analisada.
        "url": url,

        # Informa se o formato da URL é válido.
        "formato_valido": False,

        # Classificação criada pelo nosso programa.
        "status": None,

        # Código HTTP devolvido pelo servidor.
        "http_status": None,

        # Endereço final após possíveis redirecionamentos.
        "url_final": None,

        # Informa se a URL mudou durante a requisição.
        "redirecionou": False,

        # Tempo total da requisição em milissegundos.
        "tempo_ms": None,

        # Armazena uma mensagem quando ocorre erro.
        "erro": None
    }


    # --------------------------------------------------------
    # VERIFICAR VALOR AUSENTE
    # --------------------------------------------------------


    # Verifica se a URL é nula/NaN.
    if pd.isna(url):

        # Classifica como URL vazia.
        resultado["status"] = (
            "URL_VAZIA"
        )

        # Não há como continuar a validação.
        return resultado


    # Converte a URL para texto.
    url = str(url)


    # Remove espaços do começo e do final.
    url = url.strip()


    # Atualiza a URL armazenada com a versão limpa.
    resultado["url"] = url


    # --------------------------------------------------------
    # VALIDAR A SINTAXE
    # --------------------------------------------------------


    # Verifica se o endereço possui estrutura de URL HTTP/HTTPS.
    if not formato_url_valido(url):

        # Marca o formato como inválido.
        resultado["status"] = (
            "URL_INVALIDA"
        )

        # Encerra a análise desta URL.
        return resultado


    # Se chegamos aqui, o formato é válido.
    resultado["formato_valido"] = True


    # --------------------------------------------------------
    # MEDIR O TEMPO DA REQUISIÇÃO
    # --------------------------------------------------------


    # Marca o instante inicial.
    inicio = time.perf_counter()


    try:

        # ----------------------------------------------------
        # PRIMEIRA TENTATIVA: HEAD
        # ----------------------------------------------------
        #
        # HEAD normalmente devolve apenas os cabeçalhos HTTP
        # e evita baixar o conteúdo completo da página.
        # ----------------------------------------------------


        # Faz a requisição HEAD.
        response = requests.head(
            url,
            headers=HEADERS,
            timeout=TIMEOUT,
            allow_redirects=True
        )


        # ----------------------------------------------------
        # SEGUNDA TENTATIVA: GET
        # ----------------------------------------------------
        #
        # Alguns servidores não respondem corretamente ao HEAD.
        # Nestes casos tentamos GET.
        #
        # stream=True evita carregar a página inteira
        # imediatamente para a memória.
        # ----------------------------------------------------


        # Verifica códigos para os quais vale tentar GET.
        if response.status_code in (
            400,
            403,
            405,
            500,
            501
        ):

            # Fecha a resposta HEAD antes da nova tentativa.
            response.close()


            # Faz a requisição GET.
            response = requests.get(
                url,
                headers=HEADERS,
                timeout=TIMEOUT,
                allow_redirects=True,
                stream=True
            )


        # Marca o instante final.
        fim = time.perf_counter()


        # Calcula o tempo em milissegundos.
        resultado["tempo_ms"] = round(
            (fim - inicio) * 1000,
            1
        )


        # Guarda o código HTTP.
        resultado["http_status"] = (
            response.status_code
        )


        # Guarda o endereço final.
        resultado["url_final"] = (
            response.url
        )


        # Compara a URL final com a URL original.
        resultado["redirecionou"] = (
            response.url.rstrip("/")
            != url.rstrip("/")
        )


        # Converte o código HTTP para uma categoria
        # mais simples de interpretar.
        resultado["status"] = classificar_http(
            response.status_code,
            resultado["redirecionou"]
        )


        # Fecha explicitamente a resposta HTTP.
        response.close()


    # --------------------------------------------------------
    # TIMEOUT
    # --------------------------------------------------------


    # Captura especificamente estouro do tempo limite.
    except requests.exceptions.Timeout:

        # Marca o instante em que o timeout ocorreu.
        fim = time.perf_counter()


        # Calcula o tempo gasto.
        resultado["tempo_ms"] = round(
            (fim - inicio) * 1000,
            1
        )


        # Classifica o problema.
        resultado["status"] = (
            "TIMEOUT"
        )


        # Registra uma explicação.
        resultado["erro"] = (
            "O servidor excedeu o tempo limite."
        )


    # --------------------------------------------------------
    # ERRO SSL
    # --------------------------------------------------------


    # Captura problemas de certificado ou conexão SSL/TLS.
    except requests.exceptions.SSLError as erro:

        # Classifica o problema.
        resultado["status"] = (
            "ERRO_SSL"
        )


        # Guarda a mensagem original da biblioteca requests.
        resultado["erro"] = str(
            erro
        )


    # --------------------------------------------------------
    # ERRO DE CONEXÃO / DNS
    # --------------------------------------------------------


    # Captura falhas de conexão ou resolução do domínio.
    except requests.exceptions.ConnectionError as erro:

        # Classifica o problema.
        resultado["status"] = (
            "ERRO_CONEXAO"
        )


        # Guarda a mensagem técnica.
        resultado["erro"] = str(
            erro
        )


    # --------------------------------------------------------
    # OUTROS ERROS DO REQUESTS
    # --------------------------------------------------------


    # Captura outras exceções específicas da biblioteca requests.
    except requests.exceptions.RequestException as erro:

        # Classifica o problema.
        resultado["status"] = (
            "ERRO_REQUEST"
        )


        # Guarda a mensagem técnica.
        resultado["erro"] = str(
            erro
        )


    # --------------------------------------------------------
    # ERRO IMPREVISTO
    # --------------------------------------------------------


    # Captura qualquer outro erro não previsto acima.
    except Exception as erro:

        # Classifica o problema.
        resultado["status"] = (
            "ERRO_PROCESSAMENTO"
        )


        # Guarda a mensagem técnica.
        resultado["erro"] = str(
            erro
        )


    # Devolve tudo que foi descoberto sobre a URL.
    return resultado


In [ ]:

# ============================================================
# TESTAR UMA URL REAL DO DATASET
# ============================================================


# Seleciona a URL da primeira linha do Fake.br.
url_teste = df.iloc[0][
    COLUNA_URL
]


# Mostra qual URL será testada.
print(
    "URL de teste:"
)

print(
    url_teste
)


# Executa a validação.
resultado_teste = verificar_url(
    url_teste
)


# Mostra o resultado completo.
resultado_teste


In [ ]:

# ============================================================
# FUNÇÃO - OBTER URLs ÚNICAS
# ============================================================


# Recebe um DataFrame e o nome da coluna de URL.
def obter_urls_unicas(
    dataframe,
    coluna
):

    # Seleciona a coluna desejada.
    serie = dataframe[
        coluna
    ]


    # Remove valores reconhecidos pelo pandas como nulos.
    serie = serie.dropna()


    # Converte todos os valores para texto.
    serie = serie.astype(
        str
    )


    # Remove espaços do começo e do final.
    serie = serie.str.strip()


    # Remove representações textuais comuns de valor ausente.
    serie = serie[
        ~serie.str.lower().isin(
            [
                "",
                "none",
                "nan",
                "null"
            ]
        )
    ]


    # Remove URLs duplicadas.
    serie = serie.drop_duplicates()


    # Converte a Series do pandas para uma lista Python.
    return serie.tolist()


In [ ]:

# ============================================================
# PREPARAR A LISTA DE URLs DO FAKE.BR
# ============================================================


# Obtém somente URLs únicas da coluna "link".
urls_dataset = obter_urls_unicas(
    df,
    COLUNA_URL
)


# Calcula quantas linhas possuem URL após a limpeza.
quantidade_com_url = (
    df[COLUNA_URL]
    .astype(str)
    .str.strip()
    .replace(
        [
            "",
            "None",
            "nan",
            "null"
        ],
        pd.NA
    )
    .notna()
    .sum()
)


# Mostra quantas linhas possuem uma URL.
print(
    "Registros com URL:",
    quantidade_com_url
)


# Mostra quantas URLs únicas serão verificadas.
print(
    "URLs únicas para validação:",
    len(urls_dataset)
)


# Mostra quantas ocorrências repetidas existem.
print(
    "Ocorrências duplicadas de URL:",
    quantidade_com_url
    - len(urls_dataset)
)


In [ ]:

# ============================================================
# TESTE RÁPIDO COM AS PRIMEIRAS URLs
# ============================================================
#
# Antes de validar todo o dataset, verificamos apenas um
# pequeno conjunto para conferir se a Internet e o código
# estão funcionando corretamente.
# ============================================================


# Seleciona as primeiras URLs definidas em QUANTIDADE_TESTE.
urls_teste = urls_dataset[
    :QUANTIDADE_TESTE
]


# Lista que armazenará os resultados do teste.
resultados_teste = []


# Percorre as URLs de teste uma por uma.
for url in urls_teste:

    # Valida a URL atual.
    resultado = verificar_url(
        url
    )


    # Guarda o resultado.
    resultados_teste.append(
        resultado
    )


    # Mostra um resumo da URL processada.
    print(
        resultado["status"],
        resultado["http_status"],
        url[:80]
    )


In [ ]:

# ============================================================
# VISUALIZAR O RESULTADO DO TESTE RÁPIDO
# ============================================================


# Transforma os resultados do teste em DataFrame.
df_teste = pd.DataFrame(
    resultados_teste
)


# Mostra a tabela.
display(
    df_teste
)


In [ ]:

# ============================================================
# FUNÇÃO - VALIDAR UMA LISTA DE URLs EM PARALELO
# ============================================================


# Recebe uma lista de URLs.
def verificar_lista_urls(
    urls,
    max_workers=8
):

    # Lista que armazenará os resultados.
    resultados = []


    # Quantidade total de URLs.
    total = len(
        urls
    )


    # Caso a lista esteja vazia, devolve um DataFrame vazio.
    if total == 0:

        return pd.DataFrame(
            resultados
        )


    # Cria um conjunto de threads.
    with ThreadPoolExecutor(
        max_workers=max_workers
    ) as executor:

        # Envia cada URL para a função verificar_url().
        futuros = {

            executor.submit(
                verificar_url,
                url
            ): url

            for url in urls
        }


        # Contador de tarefas concluídas.
        concluido = 0


        # Recebe os resultados conforme as threads terminam.
        for futuro in as_completed(
            futuros
        ):

            # Atualiza o contador.
            concluido += 1


            try:

                # Obtém o resultado produzido pela thread.
                resultado = futuro.result()


            # Proteção adicional caso uma thread gere erro
            # fora da função verificar_url().
            except Exception as erro:

                # Monta um registro mínimo de erro.
                resultado = {

                    "url": futuros[
                        futuro
                    ],

                    "formato_valido": None,

                    "status": (
                        "ERRO_THREAD"
                    ),

                    "http_status": None,

                    "url_final": None,

                    "redirecionou": None,

                    "tempo_ms": None,

                    "erro": str(
                        erro
                    )
                }


            # Adiciona o resultado à lista.
            resultados.append(
                resultado
            )


            # Mostra o progresso a cada 100 URLs
            # ou quando a última URL terminar.
            if (
                concluido % 100 == 0
                or concluido == total
            ):

                print(
                    f"{concluido}/{total} "
                    f"URLs verificadas"
                )


    # Converte todos os resultados em DataFrame.
    return pd.DataFrame(
        resultados
    )


In [ ]:

# ============================================================
# EXECUTAR A VALIDAÇÃO COMPLETA
# ============================================================
#
# ATENÇÃO:
#
# Esta célula acessa todas as URLs únicas do dataset.
#
# O tempo total depende:
#
# - da quantidade de URLs;
# - da velocidade da Internet;
# - do número de sites que entram em timeout;
# - de bloqueios dos servidores.
#
# O valor MAX_WORKERS controla quantas verificações
# acontecem simultaneamente.
# ============================================================


# Executa a validação de todas as URLs únicas.
df_validacao = verificar_lista_urls(
    urls_dataset,
    max_workers=MAX_WORKERS
)


# Mostra a dimensão da tabela de validação.
print(
    "Dimensão da validação:",
    df_validacao.shape
)


In [ ]:

# ============================================================
# SALVAR CHECKPOINT DA VALIDAÇÃO
# ============================================================


# Salva somente o resultado técnico das URLs.
df_validacao.to_csv(
    ARQUIVO_CHECKPOINT,
    index=False,
    encoding="utf-8-sig"
)


# Confirma a criação do arquivo.
print(
    f"Checkpoint salvo: "
    f"{ARQUIVO_CHECKPOINT}"
)


In [ ]:

# ============================================================
# RESUMO DOS STATUS ENCONTRADOS
# ============================================================


# Conta quantas URLs ficaram em cada status.
resumo_status = (
    df_validacao[
        "status"
    ]
    .value_counts(
        dropna=False
    )
)


# Mostra o resumo.
display(
    resumo_status.to_frame(
        "quantidade"
    )
)


In [ ]:

# ============================================================
# RESUMO PERCENTUAL DOS STATUS
# ============================================================


# Calcula a proporção de cada status.
resumo_percentual = (
    df_validacao[
        "status"
    ]
    .value_counts(
        normalize=True,
        dropna=False
    )
    .mul(
        100
    )
    .round(
        2
    )
)


# Mostra o percentual.
display(
    resumo_percentual.to_frame(
        "percentual"
    )
)


In [ ]:

# ============================================================
# PREPARAR A VALIDAÇÃO PARA JUNTAR AO DATASET
# ============================================================


# Renomeia a coluna "url" da tabela de validação para "link".
#
# Com isso podemos usar a mesma chave existente no Fake.br.
validacao_para_merge = (
    df_validacao
    .rename(
        columns={
            "url": COLUNA_URL
        }
    )
)


# Prefixa todas as colunas de resultado com "url_",
# exceto a própria coluna "link".
validacao_para_merge = (
    validacao_para_merge
    .rename(
        columns={

            coluna:
            f"url_{coluna}"

            for coluna
            in validacao_para_merge.columns

            if coluna != COLUNA_URL
        }
    )
)


# Mostra as colunas que serão adicionadas ao dataset.
print(
    "Colunas da validação:"
)

print(
    validacao_para_merge.columns.tolist()
)


In [ ]:

# ============================================================
# JUNTAR A VALIDAÇÃO AO DATASET FAKE.BR
# ============================================================


# Faz um merge pela coluna "link".
#
# how="left" mantém todas as notícias do dataset original,
# mesmo se alguma URL não possuir resultado de validação.
df_final = df.merge(
    validacao_para_merge,
    on=COLUNA_URL,
    how="left"
)


# ============================================================
# CLASSIFICAR A SITUAÇÃO DE ACESSO DA URL
# ============================================================


# Define quais status representam acesso técnico bem-sucedido.
#
# VALIDA:
# a URL respondeu normalmente.
#
# REDIRECIONADA:
# a URL mudou de endereço, mas chegou a um destino que respondeu.
STATUS_SEM_ERRO_ACESSO = [
    "VALIDA",
    "REDIRECIONADA"
]


# Cria a nova coluna "url_situacao_acesso".
#
# Para cada linha:
#
# - se url_status estiver em STATUS_SEM_ERRO_ACESSO,
#   grava "SEM_ERRO_ACESSO";
#
# - caso contrário,
#   grava "COM_ERRO_ACESSO".
df_final[
    "url_situacao_acesso"
] = df_final[
    "url_status"
].apply(
    lambda status:
    (
        "SEM_ERRO_ACESSO"
        if status in STATUS_SEM_ERRO_ACESSO
        else "COM_ERRO_ACESSO"
    )
)


# Mostra a dimensão do dataset enriquecido.
print(
    "Dimensão do dataset final:",
    df_final.shape
)


# Mostra algumas colunas para conferência.
display(
    df_final[
        [
            "classe",
            "link",
            "url_situacao_acesso",
            "url_status",
            "url_http_status",
            "url_final",
            "url_redirecionou",
            "url_tempo_ms"
        ]
    ].head(10)
)


In [ ]:

# ============================================================
# SEPARAR O DATASET POR SITUAÇÃO DE ACESSO À URL
# ============================================================
#
# A partir desta etapa teremos três DataFrames:
#
# 1. df_final
#    contém todos os registros.
#
# 2. df_sem_erro_url
#    contém somente registros cujas URLs foram classificadas
#    como SEM_ERRO_ACESSO.
#
# 3. df_com_erro_url
#    contém somente registros cujas URLs foram classificadas
#    como COM_ERRO_ACESSO.
#
# Esta separação facilita utilizar somente as notícias
# acessíveis em etapas posteriores, sem perder os registros
# que precisam ser investigados separadamente.
# ============================================================


# Seleciona somente as linhas sem erro de acesso à URL.
df_sem_erro_url = df_final[
    df_final[
        "url_situacao_acesso"
    ] == "SEM_ERRO_ACESSO"
].copy()


# Seleciona somente as linhas com erro de acesso à URL.
df_com_erro_url = df_final[
    df_final[
        "url_situacao_acesso"
    ] == "COM_ERRO_ACESSO"
].copy()


# Mostra a quantidade total de registros.
print(
    "Total de registros:",
    len(df_final)
)


# Mostra a quantidade de registros sem erro de acesso.
print(
    "Registros SEM erro de acesso:",
    len(df_sem_erro_url)
)


# Mostra a quantidade de registros com erro de acesso.
print(
    "Registros COM erro de acesso:",
    len(df_com_erro_url)
)


# Confere se a soma dos dois conjuntos corresponde
# ao dataset completo.
print(
    "Conferência da separação:",
    (
        len(df_sem_erro_url)
        + len(df_com_erro_url)
        == len(df_final)
    )
)


# Mostra a distribuição percentual.
percentual_sem_erro = round(
    (
        len(df_sem_erro_url)
        / len(df_final)
        * 100
    ),
    2
)


# Calcula o percentual de registros com erro de acesso.
percentual_com_erro = round(
    (
        len(df_com_erro_url)
        / len(df_final)
        * 100
    ),
    2
)


# Mostra os percentuais.
print(
    f"Percentual SEM erro: "
    f"{percentual_sem_erro}%"
)

print(
    f"Percentual COM erro: "
    f"{percentual_com_erro}%"
)


In [ ]:

# ============================================================
# COMPARAR SITUAÇÃO DE ACESSO E STATUS POR CLASSE
# ============================================================
#
# Estas tabelas são apenas descritivas.
#
# Elas NÃO devem ser interpretadas como evidência de que uma
# classe é mais confiável que a outra.
#
# Sites antigos, removidos, bloqueados ou com infraestrutura
# indisponível podem afetar as duas classes por motivos
# técnicos e históricos.
# ============================================================


# Cria uma tabela cruzando a classe da notícia
# com a situação geral de acesso à URL.
situacao_por_classe = pd.crosstab(
    df_final[
        "classe"
    ],
    df_final[
        "url_situacao_acesso"
    ],
    margins=True
)


# Mostra a tabela de situação de acesso.
print(
    "Situação de acesso por classe:"
)

display(
    situacao_por_classe
)


# Cria outra tabela mais detalhada cruzando
# classe e status técnico da URL.
status_por_classe = pd.crosstab(
    df_final[
        "classe"
    ],
    df_final[
        "url_status"
    ],
    margins=True
)


# Mostra a tabela detalhada de status.
print(
    "Status técnico por classe:"
)

display(
    status_por_classe
)


In [ ]:

# ============================================================
# PESQUISA - DATA STORYTELLING E BIBLIOTECAS DE GRÁFICOS
# ============================================================
#
# Data storytelling não significa "enfeitar" gráficos.
#
# A ideia é organizar uma sequência de perguntas, evidências
# visuais e cuidados de interpretação para que o leitor entenda:
#
# - qual pergunta está sendo feita;
# - qual gráfico ajuda a responder;
# - o que os dados realmente mostram;
# - o que NÃO pode ser concluído apenas com aquele gráfico.
#
# Referências consultadas para esta etapa:
#
# Tableau - boas práticas para histórias com dados:
# https://help.tableau.com/current/pro/desktop/pt-br/story_best_practices.htm
#
# Matplotlib - documentação oficial:
# https://matplotlib.org/stable/users/getting_started/
#
# Seaborn - documentação oficial:
# https://seaborn.pydata.org/tutorial
#
# Plotly - fundamentos:
# https://plotly.com/python/plotly-fundamentals/
#
# Penn State - correlação e causalidade:
# https://online.stat.psu.edu/stat200/Lesson03
#
# Decisão para o notebook C1NC0:
#
# Utilizar principalmente Seaborn + Matplotlib.
#
# Motivos:
#
# - Seaborn facilita gráficos estatísticos diretamente
#   a partir de DataFrames do pandas;
# - Matplotlib fornece controle sobre títulos, eixos,
#   anotações e exportação em PNG;
# - as duas bibliotecas já se encaixam na EDA prevista
#   na metodologia do grupo;
# - os PNGs podem ser reutilizados em documentação
#   e apresentações.
#
# Plotly permanece como alternativa futura caso o grupo
# queira gráficos interativos com zoom, hover e HTML.
# ============================================================


# Cria uma tabela simples comparando as bibliotecas pesquisadas.
comparacao_bibliotecas = pd.DataFrame(
    [
        {
            "biblioteca": "Matplotlib",
            "tipo": "Estática / base",
            "ponto_forte": (
                "Controle detalhado e exportação "
                "para imagens"
            ),
            "uso_sugerido_C1NC0": (
                "Base dos gráficos e arquivos PNG"
            )
        },
        {
            "biblioteca": "Seaborn",
            "tipo": "Estatística",
            "ponto_forte": (
                "Integração com pandas e gráficos "
                "de distribuição/relacionamento"
            ),
            "uso_sugerido_C1NC0": (
                "EDA e data storytelling"
            )
        },
        {
            "biblioteca": "Plotly",
            "tipo": "Interativa",
            "ponto_forte": (
                "Hover, zoom e exportação HTML"
            ),
            "uso_sugerido_C1NC0": (
                "Opcional para demonstração interativa"
            )
        }
    ]
)


# Mostra a comparação.
display(
    comparacao_bibliotecas
)


# Registra a escolha atual do grupo para este notebook.
print(
    "Escolha atual para este notebook: "
    "Seaborn + Matplotlib."
)


In [ ]:

# ============================================================
# PREPARAR O DATASET PARA A ANÁLISE DE QUALIDADE
# ============================================================
#
# O DataFrame "df_final" já contém:
#
# - os metadados do Fake.br;
# - a classe fake/true;
# - o resultado da validação das URLs.
#
# Agora criaremos uma cópia para:
#
# - normalizar valores ausentes;
# - converter números;
# - converter datas;
# - extrair domínio;
# - criar features proporcionais;
# - identificar anomalias e incoerências.
# ============================================================


# Cria uma cópia para preservar a etapa anterior.
df_analise = df_final.copy()


# Lista de textos que devem ser interpretados como ausência.
VALORES_AUSENTES_TEXTO = [
    "",
    "none",
    "nan",
    "null",
    "n/a",
    "na"
]


# Colunas textuais principais que serão normalizadas.
colunas_texto = [
    "autor",
    "link",
    "categoria",
    "data_publicacao"
]


# Percorre as colunas textuais existentes.
for coluna in colunas_texto:

    # Confirma que a coluna existe no DataFrame.
    if coluna in df_analise.columns:

        # Converte para o tipo string do pandas.
        df_analise[coluna] = (
            df_analise[coluna]
            .astype("string")
            .str.strip()
        )

        # Troca representações textuais de ausência por pd.NA.
        mascara_ausente = (
            df_analise[coluna]
            .str.lower()
            .isin(
                VALORES_AUSENTES_TEXTO
            )
        )

        # Aplica a substituição.
        df_analise.loc[
            mascara_ausente,
            coluna
        ] = pd.NA


# Lista de features que devem ser numéricas.
colunas_numericas = [
    "num_tokens",
    "num_palavras",
    "num_types",
    "num_links_internos",
    "num_palavras_maiusculas",
    "num_verbos",
    "num_verbos_subjuntivo_imperativo",
    "num_substantivos",
    "num_adjetivos",
    "num_adverbios",
    "num_verbos_modais",
    "num_pronomes_singular_1_2",
    "num_pronomes_plural_1",
    "num_pronomes",
    "pausality",
    "num_caracteres",
    "comprimento_medio_sentenca",
    "comprimento_medio_palavra",
    "percentual_erros_ortograficos",
    "emotividade",
    "diversidade"
]


# Converte as colunas numéricas.
for coluna in colunas_numericas:

    # Confirma que a coluna existe.
    if coluna in df_analise.columns:

        # Valores não numéricos passam a ser NaN.
        df_analise[coluna] = pd.to_numeric(
            df_analise[coluna],
            errors="coerce"
        )


# Converte a data de publicação.
#
# errors="coerce" transforma datas não interpretáveis em NaT.
df_analise[
    "data_publicacao_parseada"
] = pd.to_datetime(
    df_analise[
        "data_publicacao"
    ],
    errors="coerce",
    dayfirst=True
)


# Função simples para extrair o domínio de uma URL.
def extrair_dominio(url):

    # Se a URL estiver ausente, devolve valor ausente.
    if pd.isna(url):

        return pd.NA


    try:

        # Interpreta a URL e obtém o domínio.
        dominio = urlparse(
            str(url)
        ).netloc.lower()


        # Remove "www." somente para facilitar agrupamentos.
        if dominio.startswith(
            "www."
        ):

            dominio = dominio[
                4:
            ]


        # Devolve o domínio.
        return dominio or pd.NA


    # Em caso de erro, devolve valor ausente.
    except Exception:

        return pd.NA


# Cria a coluna de domínio.
df_analise[
    "dominio"
] = df_analise[
    "link"
].apply(
    extrair_dominio
)


# Mostra um resumo da preparação.
print(
    "Dataset preparado para análise."
)

print(
    "Linhas:",
    len(df_analise)
)

print(
    "Colunas:",
    len(df_analise.columns)
)


In [ ]:

# ============================================================
# CRIAR FEATURES PROPORCIONAIS
# ============================================================
#
# Contagens brutas podem ser influenciadas pelo tamanho do texto.
#
# Exemplo:
#
# uma notícia com 2.000 palavras tende naturalmente a possuir
# mais verbos do que uma notícia com 200 palavras.
#
# Por isso criamos algumas taxas por 100 palavras.
#
# Isso ajuda a comparar textos de tamanhos diferentes.
# ============================================================


# Evita divisão por zero.
denominador_palavras = (
    df_analise[
        "num_palavras"
    ]
    .where(
        df_analise[
            "num_palavras"
        ] > 0
    )
)


# Dicionário:
#
# chave = nome da nova feature;
# valor = coluna de contagem original.
features_por_100 = {
    "taxa_maiusculas_100": (
        "num_palavras_maiusculas"
    ),
    "taxa_verbos_100": (
        "num_verbos"
    ),
    "taxa_substantivos_100": (
        "num_substantivos"
    ),
    "taxa_adjetivos_100": (
        "num_adjetivos"
    ),
    "taxa_adverbios_100": (
        "num_adverbios"
    ),
    "taxa_pronomes_100": (
        "num_pronomes"
    )
}


# Percorre cada feature proporcional.
for nova_feature, coluna_origem in (
    features_por_100.items()
):

    # Calcula a quantidade por 100 palavras.
    df_analise[
        nova_feature
    ] = (
        df_analise[
            coluna_origem
        ]
        / denominador_palavras
        * 100
    )


# Cria uma transformação logarítmica do tamanho do texto.
#
# log1p(x) calcula log(1 + x), permitindo valor zero.
df_analise[
    "log_num_palavras"
] = np.log1p(
    df_analise[
        "num_palavras"
    ].clip(
        lower=0
    )
)


# Mostra as novas features criadas.
print(
    "Features proporcionais criadas:"
)

print(
    list(
        features_por_100.keys()
    )
)


In [ ]:

# ============================================================
# IDENTIFICAR INCOERÊNCIAS POR REGRAS DE QUALIDADE
# ============================================================
#
# Nesta etapa usamos regras explícitas.
#
# IMPORTANTE:
#
# "incoerência" não significa automaticamente erro definitivo.
#
# A regra serve para separar registros que merecem revisão.
# ============================================================


# Identifica classe diferente de fake/true.
df_analise[
    "incoerencia_classe"
] = ~df_analise[
    "classe"
].isin(
    [
        "fake",
        "true"
    ]
)


# Identifica links duplicados.
#
# Duplicidade pode ser esperada em alguns datasets,
# portanto é registrada como sinal de qualidade e não
# como erro definitivo.
df_analise[
    "link_duplicado"
] = (
    df_analise[
        "link"
    ]
    .notna()
    & df_analise[
        "link"
    ]
    .duplicated(
        keep=False
    )
)


# Conta quantas classes diferentes aparecem para cada link.
classes_por_link = (
    df_analise
    .dropna(
        subset=[
            "link"
        ]
    )
    .groupby(
        "link"
    )[
        "classe"
    ]
    .nunique()
)


# Seleciona links que aparecem em mais de uma classe.
links_classes_conflitantes = set(
    classes_por_link[
        classes_por_link > 1
    ].index
)


# Marca links associados simultaneamente a fake e true.
df_analise[
    "incoerencia_link_classe"
] = df_analise[
    "link"
].isin(
    links_classes_conflitantes
)


# Registra ausência de autor.
df_analise[
    "autor_ausente"
] = df_analise[
    "autor"
].isna()


# Registra ausência de categoria.
df_analise[
    "categoria_ausente"
] = df_analise[
    "categoria"
].isna()


# Registra ausência da data original.
df_analise[
    "data_publicacao_ausente"
] = df_analise[
    "data_publicacao"
].isna()


# Identifica data presente que não pôde ser interpretada.
df_analise[
    "incoerencia_data_invalida"
] = (
    df_analise[
        "data_publicacao"
    ].notna()
    & df_analise[
        "data_publicacao_parseada"
    ].isna()
)


# Identifica qualquer valor negativo nas colunas numéricas
# em que uma contagem/medida negativa não é esperada.
colunas_nao_negativas = [
    coluna
    for coluna in colunas_numericas
    if coluna in df_analise.columns
]


# Cria a máscara de números negativos.
df_analise[
    "incoerencia_valor_negativo"
] = (
    df_analise[
        colunas_nao_negativas
    ]
    .lt(
        0
    )
    .any(
        axis=1
    )
)


# Número de types/palavras diferentes não deveria superar
# o número total de palavras no mesmo registro.
df_analise[
    "incoerencia_types_maior_palavras"
] = (
    df_analise[
        "num_types"
    ]
    > df_analise[
        "num_palavras"
    ]
)


# O número de caracteres não deveria ser menor que o
# número de palavras quando ambas as medidas representam
# o mesmo texto.
df_analise[
    "incoerencia_caracteres_menor_palavras"
] = (
    df_analise[
        "num_caracteres"
    ]
    < df_analise[
        "num_palavras"
    ]
)


# Lista de contagens linguísticas que individualmente
# não deveriam superar o total de palavras.
componentes_linguisticos = [
    "num_palavras_maiusculas",
    "num_verbos",
    "num_substantivos",
    "num_adjetivos",
    "num_adverbios",
    "num_pronomes"
]


# Cria uma comparação linha a linha com num_palavras.
df_analise[
    "incoerencia_componente_maior_palavras"
] = (
    df_analise[
        componentes_linguisticos
    ]
    .gt(
        df_analise[
            "num_palavras"
        ],
        axis=0
    )
    .any(
        axis=1
    )
)


# Registra URL com erro técnico de acesso.
#
# Esta coluna NÃO entra na contagem de incoerências do conteúdo.
df_analise[
    "url_com_erro_acesso"
] = (
    df_analise[
        "url_situacao_acesso"
    ]
    == "COM_ERRO_ACESSO"
)


# Lista somente das regras consideradas incoerências.
colunas_incoerencia = [
    "incoerencia_classe",
    "incoerencia_link_classe",
    "incoerencia_data_invalida",
    "incoerencia_valor_negativo",
    "incoerencia_types_maior_palavras",
    "incoerencia_caracteres_menor_palavras",
    "incoerencia_componente_maior_palavras"
]


# Conta quantas regras de incoerência cada linha acionou.
df_analise[
    "qtd_incoerencias"
] = (
    df_analise[
        colunas_incoerencia
    ]
    .sum(
        axis=1
    )
)


# Cria uma flag geral.
df_analise[
    "possui_incoerencia"
] = (
    df_analise[
        "qtd_incoerencias"
    ]
    > 0
)


# Mostra um resumo inicial.
print(
    "Registros com pelo menos uma incoerência:",
    df_analise[
        "possui_incoerencia"
    ].sum()
)

print(
    "Links duplicados:",
    df_analise[
        "link_duplicado"
    ].sum()
)

print(
    "URLs com erro de acesso:",
    df_analise[
        "url_com_erro_acesso"
    ].sum()
)


In [ ]:

# ============================================================
# IDENTIFICAR ANOMALIAS ESTATÍSTICAS COM IQR
# ============================================================
#
# IQR = intervalo interquartil.
#
# Para cada feature:
#
# Q1 = percentil 25
# Q3 = percentil 75
# IQR = Q3 - Q1
#
# Limite inferior = Q1 - 1.5 * IQR
# Limite superior = Q3 + 1.5 * IQR
#
# Valores fora dessa faixa são marcados como outliers.
#
# IMPORTANTE:
#
# Outlier NÃO significa dado errado.
#
# Pode ser:
#
# - erro;
# - caso raro legítimo;
# - comportamento extremo;
# - mudança de padrão;
# - informação interessante para investigação.
# ============================================================


# Features escolhidas para análise de outliers.
features_iqr = [
    "num_palavras",
    "num_links_internos",
    "comprimento_medio_sentenca",
    "comprimento_medio_palavra",
    "percentual_erros_ortograficos",
    "emotividade",
    "diversidade",
    "taxa_maiusculas_100",
    "taxa_verbos_100",
    "taxa_adjetivos_100",
    "taxa_adverbios_100",
    "taxa_pronomes_100"
]


# Lista para registrar os limites calculados.
limites_iqr = []


# Lista das colunas de flag que serão criadas.
colunas_outlier_iqr = []


# Percorre as features escolhidas.
for feature in features_iqr:

    # Obtém somente valores válidos.
    serie = df_analise[
        feature
    ].dropna()


    # Se houver poucos dados válidos, pula a feature.
    if len(serie) < 10:

        continue


    # Calcula o primeiro quartil.
    q1 = serie.quantile(
        0.25
    )


    # Calcula o terceiro quartil.
    q3 = serie.quantile(
        0.75
    )


    # Calcula o intervalo interquartil.
    iqr = q3 - q1


    # Calcula o limite inferior.
    limite_inferior = (
        q1
        - 1.5 * iqr
    )


    # Calcula o limite superior.
    limite_superior = (
        q3
        + 1.5 * iqr
    )


    # Define o nome da coluna de flag.
    coluna_flag = (
        f"outlier_iqr_{feature}"
    )


    # Registra o nome da coluna.
    colunas_outlier_iqr.append(
        coluna_flag
    )


    # Marca os registros fora da faixa.
    df_analise[
        coluna_flag
    ] = (
        (
            df_analise[
                feature
            ]
            < limite_inferior
        )
        |
        (
            df_analise[
                feature
            ]
            > limite_superior
        )
    ).fillna(
        False
    )


    # Guarda os limites para documentação.
    limites_iqr.append(
        {
            "feature": feature,
            "q1": q1,
            "q3": q3,
            "iqr": iqr,
            "limite_inferior": limite_inferior,
            "limite_superior": limite_superior,
            "qtd_outliers": int(
                df_analise[
                    coluna_flag
                ].sum()
            )
        }
    )


# Converte o resumo dos limites em DataFrame.
df_limites_iqr = pd.DataFrame(
    limites_iqr
)


# Conta quantas features marcaram cada registro como outlier.
df_analise[
    "qtd_outliers_iqr"
] = (
    df_analise[
        colunas_outlier_iqr
    ]
    .sum(
        axis=1
    )
)


# Cria uma flag geral de anomalia IQR.
df_analise[
    "anomalia_iqr"
] = (
    df_analise[
        "qtd_outliers_iqr"
    ]
    > 0
)


# Mostra os limites e quantidades.
display(
    df_limites_iqr
)


# Mostra a quantidade de registros marcados.
print(
    "Registros com pelo menos um outlier IQR:",
    df_analise[
        "anomalia_iqr"
    ].sum()
)


In [ ]:

# ============================================================
# IDENTIFICAR ANOMALIAS MULTIVARIADAS COM ISOLATION FOREST
# ============================================================
#
# O IQR analisa cada feature separadamente.
#
# O Isolation Forest procura combinações incomuns entre
# várias features ao mesmo tempo.
#
# Exemplo:
#
# uma notícia pode não ser extrema em nenhuma coluna isolada,
# mas a combinação entre tamanho, diversidade, emotividade
# e taxas linguísticas pode ser incomum.
#
# IMPORTANTE:
#
# A flag gerada NÃO significa "fake".
#
# Ela significa apenas:
#
# "este registro possui um perfil estatístico menos comum
# dentro das features analisadas".
# ============================================================


# Features usadas na detecção multivariada.
features_isolation = [
    "log_num_palavras",
    "num_links_internos",
    "comprimento_medio_sentenca",
    "comprimento_medio_palavra",
    "percentual_erros_ortograficos",
    "emotividade",
    "diversidade",
    "taxa_maiusculas_100",
    "taxa_verbos_100",
    "taxa_substantivos_100",
    "taxa_adjetivos_100",
    "taxa_adverbios_100",
    "taxa_pronomes_100"
]


# Cria uma cópia somente com as features escolhidas.
X_isolation = df_analise[
    features_isolation
].copy()


# Substitui infinito por ausente.
X_isolation = X_isolation.replace(
    [
        np.inf,
        -np.inf
    ],
    np.nan
)


# Preenche valores ausentes pela mediana de cada feature.
X_isolation = X_isolation.fillna(
    X_isolation.median(
        numeric_only=True
    )
)


# Caso alguma coluna permaneça totalmente vazia,
# preenche o restante com zero.
X_isolation = X_isolation.fillna(
    0
)


# Cria o modelo de detecção de anomalias.
modelo_isolation = IsolationForest(
    n_estimators=200,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)


# Ajusta o modelo e obtém:
#
#  1 = observação considerada normal;
# -1 = observação considerada anômala.
predicao_isolation = (
    modelo_isolation
    .fit_predict(
        X_isolation
    )
)


# Cria a flag booleana.
df_analise[
    "anomalia_isolation_forest"
] = (
    predicao_isolation
    == -1
)


# score_samples retorna valores de normalidade.
#
# Multiplicamos por -1 para facilitar a leitura:
# quanto maior o valor abaixo, mais atípico o perfil.
df_analise[
    "score_anomalia_isolation_forest"
] = -modelo_isolation.score_samples(
    X_isolation
)


# Combina as duas abordagens de anomalia.
df_analise[
    "possui_anomalia"
] = (
    df_analise[
        "anomalia_iqr"
    ]
    |
    df_analise[
        "anomalia_isolation_forest"
    ]
)


# Cria uma flag de revisão de qualidade.
#
# Ela NÃO inclui apenas erro de URL porque indisponibilidade
# do site é um problema técnico separado.
df_analise[
    "requer_revisao_dados"
] = (
    df_analise[
        "possui_incoerencia"
    ]
    |
    df_analise[
        "possui_anomalia"
    ]
)


# Mostra um resumo.
print(
    "Anomalias pelo Isolation Forest:",
    df_analise[
        "anomalia_isolation_forest"
    ].sum()
)

print(
    "Registros com alguma anomalia:",
    df_analise[
        "possui_anomalia"
    ].sum()
)

print(
    "Registros que requerem revisão:",
    df_analise[
        "requer_revisao_dados"
    ].sum()
)


In [ ]:

# ============================================================
# RESUMO DAS ANOMALIAS E INCOERÊNCIAS
# ============================================================


# Lista das flags que queremos resumir.
flags_qualidade = (
    colunas_incoerencia
    + [
        "link_duplicado",
        "autor_ausente",
        "categoria_ausente",
        "data_publicacao_ausente",
        "url_com_erro_acesso",
        "anomalia_iqr",
        "anomalia_isolation_forest",
        "possui_anomalia",
        "possui_incoerencia",
        "requer_revisao_dados"
    ]
)


# Lista que receberá o resumo.
resumo_qualidade = []


# Percorre cada flag.
for flag in flags_qualidade:

    # Conta os registros marcados.
    quantidade = int(
        df_analise[
            flag
        ].fillna(
            False
        ).sum()
    )


    # Calcula a porcentagem.
    percentual = round(
        quantidade
        / len(df_analise)
        * 100,
        2
    )


    # Adiciona ao resumo.
    resumo_qualidade.append(
        {
            "indicador": flag,
            "quantidade": quantidade,
            "percentual": percentual
        }
    )


# Converte em DataFrame.
df_resumo_qualidade = pd.DataFrame(
    resumo_qualidade
).sort_values(
    "quantidade",
    ascending=False
)


# Mostra o resumo.
display(
    df_resumo_qualidade
)


In [ ]:

# ============================================================
# PERGUNTAS ORIENTADORAS SOBRE OS DADOS
# ============================================================
#
# Em vez de começar pelos gráficos, começamos por perguntas.
#
# O gráfico entra para ajudar a responder uma pergunta,
# e não apenas porque "fica bonito".
#
# As perguntas abaixo fazem parte do data storytelling.
# ============================================================


# Cria uma lista de perguntas investigáveis no dataset.
perguntas_dados = [
    {
        "id": "P1",
        "pergunta": (
            "O Fake.br está balanceado entre os rótulos "
            "fake e true?"
        ),
        "evidencia": (
            "Contagem e percentual da coluna classe."
        ),
        "cuidado": (
            "Balanceamento do corpus não significa que a "
            "realidade possui 50% de cada classe."
        )
    },
    {
        "id": "P2",
        "pergunta": (
            "A proporção de URLs com erro de acesso muda "
            "entre fake e true?"
        ),
        "evidencia": (
            "Percentual de url_situacao_acesso por classe."
        ),
        "cuidado": (
            "URL fora do ar não prova falsidade; pode refletir "
            "idade, remoção, bloqueio ou mudança do site."
        )
    },
    {
        "id": "P3",
        "pergunta": (
            "Os textos rotulados como fake e true possuem "
            "diferenças de tamanho?"
        ),
        "evidencia": (
            "Distribuição e mediana de num_palavras por classe."
        ),
        "cuidado": (
            "Diferença de tamanho é associação descritiva, "
            "não causa da classe."
        )
    },
    {
        "id": "P4",
        "pergunta": (
            "A proporção do rótulo fake é igual em todas "
            "as categorias?"
        ),
        "evidencia": (
            "Cruzamento categoria x classe."
        ),
        "cuidado": (
            "Diferenças podem refletir a construção e seleção "
            "do corpus."
        )
    },
    {
        "id": "P5",
        "pergunta": (
            "Quais atributos aparecem com maior frequência "
            "entre os registros anômalos?"
        ),
        "evidencia": (
            "Flags IQR, Isolation Forest e regras de qualidade."
        ),
        "cuidado": (
            "Anomalia significa caso incomum, não notícia falsa."
        )
    },
    {
        "id": "P6",
        "pergunta": (
            "Quais variáveis numéricas se movimentam juntas?"
        ),
        "evidencia": (
            "Matriz de correlação e pares com maior "
            "correlação absoluta."
        ),
        "cuidado": (
            "Correlação não implica causalidade."
        )
    },
    {
        "id": "P7",
        "pergunta": (
            "Existem perfis de notícias semelhantes quando "
            "ignoramos o rótulo fake/true?"
        ),
        "evidencia": (
            "K-Means, silhouette e projeção PCA."
        ),
        "cuidado": (
            "Cluster não é target e não deve receber o nome "
            "de fake, true ou confiável."
        )
    }
]


# Converte as perguntas em tabela.
df_perguntas = pd.DataFrame(
    perguntas_dados
)


# Mostra o roteiro de investigação.
display(
    df_perguntas
)


In [ ]:

# ============================================================
# FUNÇÃO AUXILIAR PARA SALVAR OS GRÁFICOS
# ============================================================


# Recebe o nome do arquivo PNG.
def finalizar_grafico(
    nome_arquivo
):

    # Ajusta automaticamente margens e espaçamentos.
    plt.tight_layout()


    # Monta o caminho completo do PNG.
    caminho = os.path.join(
        PASTA_GRAFICOS,
        nome_arquivo
    )


    # Salva o gráfico em resolução adequada para apresentação.
    plt.savefig(
        caminho,
        dpi=160,
        bbox_inches="tight"
    )


    # Mostra o gráfico no notebook.
    plt.show()


    # Fecha a figura para liberar memória.
    plt.close()


    # Devolve o caminho gerado.
    return caminho


In [ ]:

# ============================================================
# DATA STORYTELLING - GRÁFICO 1
# PERGUNTA: COMO O CORPUS ESTÁ DISTRIBUÍDO ENTRE FAKE E TRUE?
# ============================================================


# Conta cada classe.
contagem_classes = (
    df_analise[
        "classe"
    ]
    .value_counts()
    .rename_axis(
        "classe"
    )
    .reset_index(
        name="quantidade"
    )
)


# Cria o gráfico.
plt.figure(
    figsize=(
        7,
        4
    )
)


# Desenha as barras com Matplotlib.
#
# Não definimos cores manualmente para manter
# o padrão visual da própria biblioteca.
plt.bar(
    contagem_classes[
        "classe"
    ],
    contagem_classes[
        "quantidade"
    ]
)


# Define título como uma conclusão descritiva neutra.
plt.title(
    "Quantidade de registros por rótulo no Fake.br"
)


# Nomeia os eixos.
plt.xlabel(
    "Rótulo do corpus"
)

plt.ylabel(
    "Quantidade de registros"
)


# Salva e mostra.
finalizar_grafico(
    "01_distribuicao_classes.png"
)


# Mostra também os percentuais.
percentual_classes = (
    df_analise[
        "classe"
    ]
    .value_counts(
        normalize=True
    )
    .mul(
        100
    )
    .round(
        2
    )
)


display(
    percentual_classes.to_frame(
        "percentual"
    )
)


# Cuidado de interpretação.
print(
    "CUIDADO: a proporção do corpus descreve o dataset; "
    "não descreve necessariamente a proporção de notícias "
    "fake/true no mundo real."
)


In [ ]:

# ============================================================
# DATA STORYTELLING - GRÁFICO 2
# PERGUNTA: O ACESSO ÀS URLs MUDA ENTRE AS CLASSES?
# ============================================================


# Cria uma tabela percentual por classe.
acesso_por_classe_pct = (
    pd.crosstab(
        df_analise[
            "classe"
        ],
        df_analise[
            "url_situacao_acesso"
        ],
        normalize="index"
    )
    .mul(
        100
    )
)


# Cria o gráfico diretamente pelo pandas.
#
# O pandas usa Matplotlib como base.
acesso_por_classe_pct.plot(
    kind="bar",
    figsize=(
        8,
        5
    )
)


# Define título e eixos.
plt.title(
    "Situação técnica das URLs dentro de cada classe"
)

plt.xlabel(
    "Rótulo do corpus"
)

plt.ylabel(
    "Percentual dentro da classe"
)


# Mantém os nomes do eixo x horizontais.
plt.xticks(
    rotation=0
)


# Salva e mostra.
finalizar_grafico(
    "02_acesso_urls_por_classe.png"
)


# Mostra os números usados no gráfico.
display(
    acesso_por_classe_pct.round(
        2
    )
)


# Cuidado de interpretação.
print(
    "CUIDADO: diferença de disponibilidade de URL pode ser "
    "causada por idade, remoção, bloqueios ou mudanças dos sites. "
    "Não é prova de falsidade."
)


In [ ]:

# ============================================================
# DATA STORYTELLING - GRÁFICO 3
# PERGUNTA: O TAMANHO DOS TEXTOS MUDA ENTRE FAKE E TRUE?
# ============================================================


# Cria a figura.
plt.figure(
    figsize=(
        8,
        5
    )
)


# Boxplot sem desenhar os pontos extremos nesta visualização.
#
# Os outliers continuam preservados no dataset e foram
# analisados separadamente.
sns.boxplot(
    data=df_analise,
    x="classe",
    y="num_palavras",
    showfliers=False
)


# Define título.
plt.title(
    "Distribuição do número de palavras por rótulo"
)


# Define eixos.
plt.xlabel(
    "Rótulo do corpus"
)

plt.ylabel(
    "Número de palavras"
)


# Salva e mostra.
finalizar_grafico(
    "03_num_palavras_por_classe.png"
)


# Calcula estatísticas por classe.
resumo_tamanho = (
    df_analise
    .groupby(
        "classe"
    )[
        "num_palavras"
    ]
    .agg(
        [
            "count",
            "mean",
            "median"
        ]
    )
    .round(
        2
    )
)


# Mostra a tabela.
display(
    resumo_tamanho
)


# Cuidado de interpretação.
print(
    "CUIDADO: se houver diferença entre as classes, isso "
    "representa uma associação no corpus. Não significa que "
    "o tamanho do texto cause falsidade ou veracidade."
)


In [ ]:

# ============================================================
# DATA STORYTELLING - GRÁFICO 4
# PERGUNTA: COMO OS RÓTULOS SE DISTRIBUEM POR CATEGORIA?
# ============================================================


# Cria uma tabela cruzando categoria e classe.
categoria_classe = pd.crosstab(
    df_analise[
        "categoria"
    ],
    df_analise[
        "classe"
    ]
)


# Garante as colunas fake e true caso existam.
for classe_esperada in [
    "fake",
    "true"
]:

    if classe_esperada not in categoria_classe.columns:

        categoria_classe[
            classe_esperada
        ] = 0


# Calcula o total por categoria.
categoria_classe[
    "total"
] = categoria_classe.sum(
    axis=1
)


# Calcula a proporção de rótulo fake dentro de cada categoria.
categoria_classe[
    "percentual_fake"
] = (
    categoria_classe[
        "fake"
    ]
    / categoria_classe[
        "total"
    ]
    * 100
)


# Ordena pela quantidade total.
categoria_story = (
    categoria_classe
    .sort_values(
        "total",
        ascending=False
    )
    .reset_index()
)


# Cria o gráfico de volume por categoria.
plt.figure(
    figsize=(
        10,
        6
    )
)


# Desenha as barras horizontais.
plt.barh(
    categoria_story[
        "categoria"
    ],
    categoria_story[
        "total"
    ]
)


# Inverte o eixo para colocar a maior categoria no topo.
plt.gca().invert_yaxis()


# Define título e eixos.
plt.title(
    "Quantidade de notícias por categoria no Fake.br"
)

plt.xlabel(
    "Quantidade de registros"
)

plt.ylabel(
    "Categoria"
)


# Salva e mostra.
finalizar_grafico(
    "04_volume_por_categoria.png"
)


# Mostra a tabela incluindo percentual de fake.
display(
    categoria_story[
        [
            "categoria",
            "fake",
            "true",
            "total",
            "percentual_fake"
        ]
    ].round(
        2
    )
)


# Cuidado de interpretação.
print(
    "CUIDADO: diferenças por categoria podem refletir como "
    "o Fake.br foi construído e amostrado. Não devem ser "
    "generalizadas automaticamente para a realidade."
)


In [ ]:

# ============================================================
# DATA STORYTELLING - GRÁFICO 4B
# PERGUNTA: QUAL É A PROPORÇÃO DO RÓTULO FAKE POR CATEGORIA?
# ============================================================
#
# Este gráfico responde diretamente a uma das perguntas
# propostas para a entrega da semana.
#
# IMPORTANTE:
#
# Estamos descrevendo os rótulos existentes no Fake.br-Corpus.
#
# Não estamos afirmando que a proporção observada no dataset
# representa a proporção real de notícias falsas em cada tema.
# ============================================================


# Ordena as categorias pelo percentual do rótulo fake.
categoria_fake_story = (
    categoria_story
    .sort_values(
        "percentual_fake",
        ascending=True
    )
)


# Cria a figura.
plt.figure(
    figsize=(
        10,
        6
    )
)


# Desenha o percentual do rótulo fake.
plt.barh(
    categoria_fake_story[
        "categoria"
    ],
    categoria_fake_story[
        "percentual_fake"
    ]
)


# Define título e eixos.
plt.title(
    "Percentual do rótulo fake dentro de cada categoria"
)

plt.xlabel(
    "Percentual de registros rotulados como fake"
)

plt.ylabel(
    "Categoria"
)


# Salva e mostra.
finalizar_grafico(
    "04b_percentual_fake_por_categoria.png"
)


# Mostra a base numérica.
display(
    categoria_fake_story[
        [
            "categoria",
            "fake",
            "true",
            "total",
            "percentual_fake"
        ]
    ].round(
        2
    )
)


# Reforça a interpretação adequada.
print(
    "CUIDADO: este gráfico descreve o Fake.br-Corpus. "
    "Ele não mede a frequência real de notícias falsas "
    "por categoria fora do dataset."
)


In [ ]:

# ============================================================
# DATA STORYTELLING - GRÁFICO 5
# PERGUNTA: ONDE APARECEM MAIS ANOMALIAS E INCOERÊNCIAS?
# ============================================================


# Calcula percentuais das três flags principais por classe.
qualidade_por_classe = (
    df_analise
    .groupby(
        "classe"
    )[
        [
            "possui_incoerencia",
            "anomalia_iqr",
            "anomalia_isolation_forest"
        ]
    ]
    .mean()
    .mul(
        100
    )
)


# Cria o gráfico usando pandas/Matplotlib.
qualidade_por_classe.plot(
    kind="bar",
    figsize=(
        10,
        5
    )
)


# Título e eixos.
plt.title(
    "Percentual de flags de qualidade por classe"
)

plt.xlabel(
    "Rótulo do corpus"
)

plt.ylabel(
    "Percentual de registros"
)


# Mantém os nomes do eixo x horizontais.
plt.xticks(
    rotation=0
)


# Salva e mostra.
finalizar_grafico(
    "05_anomalias_incoerencias_por_classe.png"
)


# Mostra os valores exatos.
display(
    qualidade_por_classe.round(
        2
    )
)


# Cuidado de interpretação.
print(
    "CUIDADO: anomalia ou incoerência é um sinal para revisão. "
    "Não é um classificador de fake news."
)


In [ ]:

# ============================================================
# DATA STORYTELLING - GRÁFICO 6
# PERGUNTA: QUAIS DOMÍNIOS APARECEM MAIS E COMO ESTÁ O ACESSO?
# ============================================================


# Remove registros sem domínio.
df_dominios = df_analise.dropna(
    subset=[
        "dominio"
    ]
).copy()


# Obtém os 15 domínios mais frequentes.
top_dominios = (
    df_dominios[
        "dominio"
    ]
    .value_counts()
    .head(
        15
    )
    .index
)


# Filtra somente esses domínios.
df_top_dominios = df_dominios[
    df_dominios[
        "dominio"
    ].isin(
        top_dominios
    )
].copy()


# Calcula quantidade e percentual de erro por domínio.
resumo_dominios = (
    df_top_dominios
    .groupby(
        "dominio"
    )
    .agg(
        quantidade=(
            "link",
            "size"
        ),
        percentual_erro_url=(
            "url_com_erro_acesso",
            "mean"
        )
    )
    .reset_index()
)


# Converte o percentual de 0-1 para 0-100.
resumo_dominios[
    "percentual_erro_url"
] = (
    resumo_dominios[
        "percentual_erro_url"
    ]
    * 100
)


# Ordena pelo volume.
resumo_dominios = resumo_dominios.sort_values(
    "quantidade",
    ascending=False
)


# Cria gráfico de quantidade.
plt.figure(
    figsize=(
        10,
        7
    )
)


# Desenha as barras horizontais.
plt.barh(
    resumo_dominios[
        "dominio"
    ],
    resumo_dominios[
        "quantidade"
    ]
)


# Coloca o domínio mais frequente no topo.
plt.gca().invert_yaxis()


# Define título.
plt.title(
    "Domínios mais frequentes no Fake.br"
)

plt.xlabel(
    "Quantidade de registros"
)

plt.ylabel(
    "Domínio"
)


# Salva e mostra.
finalizar_grafico(
    "06_top_dominios.png"
)


# Mostra a tabela com erro de acesso.
display(
    resumo_dominios.round(
        2
    )
)


# Cuidado de interpretação.
print(
    "CUIDADO: frequência de domínio e disponibilidade técnica "
    "não equivalem a credibilidade editorial."
)


In [ ]:

# ============================================================
# CORRELAÇÃO - NÃO CONFUNDIR COM CAUSALIDADE
# ============================================================
#
# Correlação mede associação entre variáveis.
#
# Ela NÃO prova que uma variável causa a outra.
#
# Exemplo:
#
# num_palavras e num_caracteres devem ter alta correlação
# porque textos maiores tendem a possuir mais dos dois.
#
# Isso não significa que aumentar caracteres "cause"
# determinada classe da notícia.
#
# Uma terceira variável, a forma de coleta, o gênero textual
# ou o próprio desenho do dataset pode explicar associações.
#
# Referência:
# https://online.stat.psu.edu/stat200/Lesson03
# ============================================================


# Features selecionadas para uma matriz interpretável.
features_correlacao = [
    "num_palavras",
    "num_types",
    "num_links_internos",
    "comprimento_medio_sentenca",
    "comprimento_medio_palavra",
    "percentual_erros_ortograficos",
    "emotividade",
    "diversidade",
    "taxa_maiusculas_100",
    "taxa_verbos_100",
    "taxa_substantivos_100",
    "taxa_adjetivos_100",
    "taxa_adverbios_100",
    "taxa_pronomes_100"
]


# Calcula a correlação de Pearson.
matriz_correlacao = (
    df_analise[
        features_correlacao
    ]
    .corr()
)


# Cria a figura.
plt.figure(
    figsize=(
        13,
        10
    )
)


# Desenha o mapa de calor.
sns.heatmap(
    matriz_correlacao,
    annot=False,
    center=0
)


# Define o título.
plt.title(
    "Matriz de correlação entre features selecionadas"
)


# Salva e mostra.
finalizar_grafico(
    "07_matriz_correlacao.png"
)


# Cria uma máscara para remover a diagonal e duplicações.
mascara_superior = np.triu(
    np.ones_like(
        matriz_correlacao,
        dtype=bool
    ),
    k=1
)


# Converte a parte superior da matriz em lista de pares.
pares_correlacao = (
    matriz_correlacao
    .where(
        mascara_superior
    )
    .stack()
    .reset_index()
)


# Renomeia as colunas.
pares_correlacao.columns = [
    "feature_1",
    "feature_2",
    "correlacao"
]


# Cria a correlação absoluta apenas para ordenar.
pares_correlacao[
    "correlacao_absoluta"
] = pares_correlacao[
    "correlacao"
].abs()


# Seleciona as 15 maiores correlações absolutas.
top_correlacoes = (
    pares_correlacao
    .sort_values(
        "correlacao_absoluta",
        ascending=False
    )
    .head(
        15
    )
)


# Mostra os pares.
display(
    top_correlacoes.round(
        3
    )
)


# Reforça a limitação.
print(
    "LEMBRETE: correlação não implica causalidade. "
    "Use os pares acima para formular novas perguntas, "
    "não para afirmar causa e efeito."
)


In [ ]:

# ============================================================
# ENCONTRAR PADRÕES SEM USAR O RÓTULO - K-MEANS
# ============================================================
#
# O objetivo aqui é verificar se os registros formam
# perfis semelhantes sem informar fake/true ao algoritmo.
#
# A classe NÃO entra nas features do K-Means.
#
# Depois do clustering podemos cruzar cluster e classe
# somente para descrever o que aconteceu.
#
# Nunca devemos chamar automaticamente:
#
# cluster 0 = fake
# cluster 1 = true
#
# Os números dos clusters são apenas identificadores.
# ============================================================


# Features escolhidas para clustering.
features_cluster = [
    "log_num_palavras",
    "num_links_internos",
    "comprimento_medio_sentenca",
    "comprimento_medio_palavra",
    "percentual_erros_ortograficos",
    "emotividade",
    "diversidade",
    "taxa_maiusculas_100",
    "taxa_verbos_100",
    "taxa_substantivos_100",
    "taxa_adjetivos_100",
    "taxa_adverbios_100",
    "taxa_pronomes_100"
]


# Seleciona as features.
X_cluster = df_analise[
    features_cluster
].copy()


# Remove infinitos.
X_cluster = X_cluster.replace(
    [
        np.inf,
        -np.inf
    ],
    np.nan
)


# Preenche ausentes com a mediana.
X_cluster = X_cluster.fillna(
    X_cluster.median(
        numeric_only=True
    )
)


# Completa alguma coluna totalmente vazia com zero.
X_cluster = X_cluster.fillna(
    0
)


# Padroniza as escalas.
#
# Isso evita que uma feature com valores numericamente maiores
# domine a distância utilizada pelo K-Means.
scaler_cluster = StandardScaler()


# Ajusta e transforma os dados.
X_cluster_padronizado = (
    scaler_cluster
    .fit_transform(
        X_cluster
    )
)


# Lista para armazenar a avaliação de cada valor de k.
avaliacao_k = []


# Testa entre 2 e 5 clusters.
for k in range(
    2,
    6
):

    # Cria o K-Means.
    modelo_k = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )


    # Ajusta e obtém os clusters.
    rotulos_k = modelo_k.fit_predict(
        X_cluster_padronizado
    )


    # Usa no máximo 2.000 registros no cálculo da silhouette
    # para manter a execução leve no Colab.
    tamanho_amostra = min(
        2000,
        len(
            X_cluster_padronizado
        )
    )


    # Calcula a silhouette.
    silhouette = silhouette_score(
        X_cluster_padronizado,
        rotulos_k,
        sample_size=tamanho_amostra,
        random_state=42
    )


    # Registra o resultado.
    avaliacao_k.append(
        {
            "k": k,
            "silhouette": silhouette,
            "inercia": modelo_k.inertia_
        }
    )


# Converte a avaliação em DataFrame.
df_avaliacao_k = pd.DataFrame(
    avaliacao_k
)


# Escolhe o k com maior silhouette.
melhor_k = int(
    df_avaliacao_k
    .sort_values(
        "silhouette",
        ascending=False
    )
    .iloc[
        0
    ][
        "k"
    ]
)


# Mostra os resultados.
display(
    df_avaliacao_k.round(
        4
    )
)


# Informa o k escolhido pelo critério.
print(
    "k selecionado pela maior silhouette:",
    melhor_k
)


In [ ]:

# ============================================================
# AJUSTAR O K-MEANS FINAL E DESCREVER OS CLUSTERS
# ============================================================


# Cria o modelo com o k escolhido.
modelo_kmeans = KMeans(
    n_clusters=melhor_k,
    random_state=42,
    n_init=10
)


# Ajusta o modelo e grava o cluster de cada linha.
df_analise[
    "cluster_exploratorio"
] = modelo_kmeans.fit_predict(
    X_cluster_padronizado
)


# Cria um perfil médio das features em cada cluster.
perfil_clusters = (
    df_analise
    .groupby(
        "cluster_exploratorio"
    )[
        features_cluster
    ]
    .mean()
    .round(
        3
    )
)


# Adiciona a quantidade de registros em cada cluster.
perfil_clusters[
    "quantidade"
] = (
    df_analise[
        "cluster_exploratorio"
    ]
    .value_counts()
    .sort_index()
)


# Calcula a proporção do rótulo fake por cluster.
#
# IMPORTANTE:
#
# esta informação NÃO foi usada para criar os clusters.
proporcao_fake_cluster = (
    df_analise
    .assign(
        classe_fake=(
            df_analise[
                "classe"
            ]
            == "fake"
        )
    )
    .groupby(
        "cluster_exploratorio"
    )[
        "classe_fake"
    ]
    .mean()
    .mul(
        100
    )
)


# Adiciona ao perfil somente para interpretação descritiva.
perfil_clusters[
    "percentual_rotulo_fake"
] = proporcao_fake_cluster.round(
    2
)


# Mostra o perfil.
display(
    perfil_clusters
)


# Reforça a interpretação correta.
print(
    "Os clusters representam perfis de features. "
    "Eles não são classes de confiabilidade."
)


In [ ]:

# ============================================================
# PCA - VISUALIZAR OS PADRÕES ENCONTRADOS PELO K-MEANS
# ============================================================
#
# O PCA reduz várias dimensões para duas componentes
# usadas somente para facilitar a visualização.
#
# O PCA não substitui o K-Means.
# ============================================================


# Cria o PCA com duas componentes.
pca = PCA(
    n_components=2
)


# Ajusta e transforma os dados padronizados.
componentes = pca.fit_transform(
    X_cluster_padronizado
)


# Guarda as duas componentes no DataFrame.
df_analise[
    "pca_1"
] = componentes[
    :,
    0
]


df_analise[
    "pca_2"
] = componentes[
    :,
    1
]


# Calcula a variância explicada por cada componente.
variancia_pca = (
    pca.explained_variance_ratio_
    * 100
)


# Seleciona uma amostra para não poluir o gráfico.
tamanho_amostra_pca = min(
    3000,
    len(df_analise)
)


# Faz a amostra de forma reproduzível.
df_pca_plot = df_analise.sample(
    n=tamanho_amostra_pca,
    random_state=42
)


# Cria a figura.
plt.figure(
    figsize=(
        9,
        6
    )
)


# Desenha a projeção colorida pelo cluster.
sns.scatterplot(
    data=df_pca_plot,
    x="pca_1",
    y="pca_2",
    hue="cluster_exploratorio",
    alpha=0.6
)


# Define título.
plt.title(
    "Projeção PCA dos padrões encontrados pelo K-Means"
)


# Nomeia os eixos incluindo a variância explicada.
plt.xlabel(
    f"PCA 1 ({variancia_pca[0]:.1f}% da variância)"
)

plt.ylabel(
    f"PCA 2 ({variancia_pca[1]:.1f}% da variância)"
)


# Salva e mostra.
finalizar_grafico(
    "08_pca_clusters.png"
)


# Mostra o total explicado pelas duas componentes.
print(
    "Variância explicada pelas duas componentes:",
    round(
        variancia_pca.sum(),
        2
    ),
    "%"
)


# Cuidado de interpretação.
print(
    "CUIDADO: proximidade visual no PCA ajuda a enxergar padrões, "
    "mas não transforma os clusters em categorias verdadeiras."
)


In [ ]:

# ============================================================
# DATA STORYTELLING - CONSTRUIR UMA NARRATIVA COM EVIDÊNCIAS
# ============================================================
#
# Esta etapa resume os gráficos em uma sequência:
#
# 1. contexto do corpus;
# 2. qualidade/acessibilidade;
# 3. diferenças descritivas;
# 4. anomalias;
# 5. relações entre features;
# 6. padrões não supervisionados;
# 7. limites de interpretação.
#
# O objetivo é apoiar a apresentação do grupo.
# ============================================================


# Obtém percentuais de erro de URL por classe.
erro_url_por_classe = (
    df_analise
    .groupby(
        "classe"
    )[
        "url_com_erro_acesso"
    ]
    .mean()
    .mul(
        100
    )
    .round(
        2
    )
)


# Obtém medianas de palavras por classe.
mediana_palavras_classe = (
    df_analise
    .groupby(
        "classe"
    )[
        "num_palavras"
    ]
    .median()
    .round(
        2
    )
)


# Obtém percentuais de anomalia por classe.
anomalia_por_classe = (
    df_analise
    .groupby(
        "classe"
    )[
        "possui_anomalia"
    ]
    .mean()
    .mul(
        100
    )
    .round(
        2
    )
)


# Define um texto para a maior correlação encontrada.
if len(
    top_correlacoes
) > 0:

    # Seleciona o primeiro par.
    maior_correlacao = top_correlacoes.iloc[
        0
    ]


    # Monta uma descrição.
    texto_maior_correlacao = (
        f"{maior_correlacao['feature_1']} x "
        f"{maior_correlacao['feature_2']} = "
        f"{maior_correlacao['correlacao']:.3f}"
    )


# Caso não exista correlação calculável.
else:

    texto_maior_correlacao = (
        "Não foi possível calcular."
    )


# Cria a narrativa em etapas.
storytelling = [
    {
        "etapa": 1,
        "pergunta": (
            "Quem compõe o dataset?"
        ),
        "evidencia": (
            f"{len(df_analise):,} registros; "
            f"classes: "
            f"{df_analise['classe'].value_counts().to_dict()}."
        ),
        "grafico": (
            "01_distribuicao_classes.png"
        ),
        "cuidado": (
            "Distribuição do corpus não representa "
            "automaticamente o mundo real."
        )
    },
    {
        "etapa": 2,
        "pergunta": (
            "Conseguimos acessar as URLs atualmente?"
        ),
        "evidencia": (
            f"Percentual com erro por classe: "
            f"{erro_url_por_classe.to_dict()}."
        ),
        "grafico": (
            "02_acesso_urls_por_classe.png"
        ),
        "cuidado": (
            "Indisponibilidade não prova falsidade."
        )
    },
    {
        "etapa": 3,
        "pergunta": (
            "O tamanho dos textos muda entre os rótulos?"
        ),
        "evidencia": (
            f"Mediana de palavras por classe: "
            f"{mediana_palavras_classe.to_dict()}."
        ),
        "grafico": (
            "03_num_palavras_por_classe.png"
        ),
        "cuidado": (
            "Diferença descritiva não é causalidade."
        )
    },
    {
        "etapa": 4,
        "pergunta": (
            "Onde aparecem casos atípicos?"
        ),
        "evidencia": (
            f"Percentual de anomalias por classe: "
            f"{anomalia_por_classe.to_dict()}."
        ),
        "grafico": (
            "05_anomalias_incoerencias_por_classe.png"
        ),
        "cuidado": (
            "Anomalia é um sinal para investigação, "
            "não um rótulo de fake."
        )
    },
    {
        "etapa": 5,
        "pergunta": (
            "Quais features apresentam associação?"
        ),
        "evidencia": (
            f"Maior correlação absoluta observada: "
            f"{texto_maior_correlacao}."
        ),
        "grafico": (
            "07_matriz_correlacao.png"
        ),
        "cuidado": (
            "Correlação não implica causalidade."
        )
    },
    {
        "etapa": 6,
        "pergunta": (
            "Existem perfis semelhantes sem usar fake/true?"
        ),
        "evidencia": (
            f"K-Means selecionou k={melhor_k} "
            f"pela maior silhouette entre 2 e 5."
        ),
        "grafico": (
            "08_pca_clusters.png"
        ),
        "cuidado": (
            "Clusters não são classes de confiabilidade."
        )
    }
]


# Converte o roteiro em DataFrame.
df_storytelling = pd.DataFrame(
    storytelling
)


# Mostra a narrativa.
display(
    df_storytelling
)


In [ ]:

# ============================================================
# PREPARAR OS DATASETS FINAIS DEPOIS DA ANÁLISE
# ============================================================
#
# Substituímos df_final pela versão enriquecida.
#
# Assim, os arquivos finais também recebem:
#
# - domínio;
# - features proporcionais;
# - flags de incoerência;
# - flags de anomalia;
# - score do Isolation Forest;
# - cluster exploratório;
# - componentes PCA.
# ============================================================


# Atualiza o DataFrame final.
df_final = df_analise.copy()


# Recria o conjunto sem erro de acesso já com as novas colunas.
df_sem_erro_url = df_final[
    df_final[
        "url_situacao_acesso"
    ] == "SEM_ERRO_ACESSO"
].copy()


# Recria o conjunto com erro de acesso já com as novas colunas.
df_com_erro_url = df_final[
    df_final[
        "url_situacao_acesso"
    ] == "COM_ERRO_ACESSO"
].copy()


# Cria o conjunto de registros para revisão de qualidade.
df_revisao_dados = df_final[
    df_final[
        "requer_revisao_dados"
    ]
].copy()


# Mostra os tamanhos.
print(
    "Dataset completo:",
    len(df_final)
)

print(
    "Sem erro de acesso:",
    len(df_sem_erro_url)
)

print(
    "Com erro de acesso:",
    len(df_com_erro_url)
)

print(
    "Com anomalia/incoerência para revisão:",
    len(df_revisao_dados)
)


In [ ]:

# ============================================================
# SALVAR OS DATASETS E RESUMOS GERADOS
# ============================================================
#
# Nesta versão serão produzidos:
#
# - dataset completo enriquecido;
# - dataset sem erro de acesso;
# - dataset com erro de acesso;
# - registros com anomalias/incoerências;
# - resumo das flags de qualidade;
# - perfil dos clusters;
# - roteiro de data storytelling;
# - ZIP com os gráficos.
# ============================================================


# Salva o dataset completo.
df_final.to_csv(
    ARQUIVO_FINAL,
    index=False,
    encoding="utf-8-sig"
)


# Salva somente registros sem erro de acesso.
df_sem_erro_url.to_csv(
    ARQUIVO_SEM_ERRO,
    index=False,
    encoding="utf-8-sig"
)


# Salva somente registros com erro de acesso.
df_com_erro_url.to_csv(
    ARQUIVO_COM_ERRO,
    index=False,
    encoding="utf-8-sig"
)


# Salva registros com anomalia ou incoerência.
df_revisao_dados.to_csv(
    ARQUIVO_REVISAO,
    index=False,
    encoding="utf-8-sig"
)


# Salva o resumo das flags.
df_resumo_qualidade.to_csv(
    ARQUIVO_RESUMO_QUALIDADE,
    index=False,
    encoding="utf-8-sig"
)


# Salva o perfil dos clusters.
perfil_clusters.to_csv(
    ARQUIVO_CLUSTERS,
    encoding="utf-8-sig"
)


# Salva o roteiro de storytelling.
df_storytelling.to_csv(
    ARQUIVO_STORYTELLING,
    index=False,
    encoding="utf-8-sig"
)


# Compacta todos os PNGs da pasta de gráficos.
shutil.make_archive(
    ARQUIVO_GRAFICOS_ZIP.replace(
        ".zip",
        ""
    ),
    "zip",
    PASTA_GRAFICOS
)


# Confirma os principais arquivos.
print(
    "Arquivos de análise criados com sucesso."
)

print(
    ARQUIVO_FINAL
)

print(
    ARQUIVO_SEM_ERRO
)

print(
    ARQUIVO_COM_ERRO
)

print(
    ARQUIVO_REVISAO
)

print(
    ARQUIVO_RESUMO_QUALIDADE
)

print(
    ARQUIVO_CLUSTERS
)

print(
    ARQUIVO_STORYTELLING
)

print(
    ARQUIVO_GRAFICOS_ZIP
)


In [ ]:

# ============================================================
# RESUMO FINAL
# ============================================================


# Mostra o título do resumo.
print(
    "=" * 78
)

print(
    "RESUMO - VALIDAÇÃO, QUALIDADE, PADRÕES E DATA STORYTELLING"
)

print(
    "=" * 78
)


# Mostra a quantidade total de registros.
print(
    "Notícias no dataset:",
    len(df_final)
)


# Mostra URLs únicas verificadas.
print(
    "URLs únicas verificadas:",
    len(df_validacao)
)


# Mostra os conjuntos por situação de acesso.
print(
    "Registros SEM erro de acesso:",
    len(df_sem_erro_url)
)

print(
    "Registros COM erro de acesso:",
    len(df_com_erro_url)
)


# Mostra indicadores de qualidade.
print(
    "Registros com incoerência:",
    int(
        df_final[
            "possui_incoerencia"
        ].sum()
    )
)

print(
    "Registros com anomalia:",
    int(
        df_final[
            "possui_anomalia"
        ].sum()
    )
)

print(
    "Registros separados para revisão:",
    len(df_revisao_dados)
)


# Mostra o resultado do clustering.
print(
    "Quantidade de clusters escolhida:",
    melhor_k
)


# Mostra a variância explicada pelo PCA.
print(
    "Variância explicada por PCA1 + PCA2:",
    round(
        variancia_pca.sum(),
        2
    ),
    "%"
)


# Mostra quantos gráficos foram gerados.
quantidade_graficos = len(
    [
        nome
        for nome in os.listdir(
            PASTA_GRAFICOS
        )
        if nome.lower().endswith(
            ".png"
        )
    ]
)


print(
    "Gráficos de storytelling gerados:",
    quantidade_graficos
)


# Reforça os limites metodológicos.
print(
    "\nINTERPRETAÇÃO:"
)

print(
    "- URL acessível não significa notícia confiável."
)

print(
    "- URL inacessível não significa notícia falsa."
)

print(
    "- Anomalia não significa fake."
)

print(
    "- Cluster não é target."
)

print(
    "- Correlação não implica causalidade."
)

print(
    "- Os rótulos fake/true descrevem o Fake.br-Corpus."
)


In [ ]:

# ============================================================
# BAIXAR OS ARQUIVOS GERADOS
# ============================================================
#
# Os downloads abaixo incluem:
#
# 1. validação técnica das URLs;
# 2. dataset completo enriquecido;
# 3. dataset sem erro de acesso;
# 4. dataset com erro de acesso;
# 5. anomalias/incoerências;
# 6. resumo de qualidade;
# 7. perfil dos clusters;
# 8. roteiro de storytelling;
# 9. gráficos em ZIP.
# ============================================================


# Baixa o checkpoint das URLs.
files.download(
    ARQUIVO_CHECKPOINT
)


# Baixa o dataset completo.
files.download(
    ARQUIVO_FINAL
)


# Baixa somente URLs sem erro.
files.download(
    ARQUIVO_SEM_ERRO
)


# Baixa somente URLs com erro.
files.download(
    ARQUIVO_COM_ERRO
)


# Baixa registros para revisão.
files.download(
    ARQUIVO_REVISAO
)


# Baixa o resumo de qualidade.
files.download(
    ARQUIVO_RESUMO_QUALIDADE
)


# Baixa o perfil de clusters.
files.download(
    ARQUIVO_CLUSTERS
)


# Baixa o roteiro de storytelling.
files.download(
    ARQUIVO_STORYTELLING
)


# Baixa os gráficos compactados.
files.download(
    ARQUIVO_GRAFICOS_ZIP
)
